<div style="border-left:14px solid #708090;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#708090;letter-spacing:.08em">WELCOME</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Start here</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">How this session works, and a check that your setup runs.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 1 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

# Computational Approaches to Cultural Heritage Image Collections

This workshop is about finding, reading and working with digitised cultural heritage **programmatically**. Drawing on the idea of *distant reading*, it asks how algorithms can "read" collections of images and their descriptions at scale, and what they miss.

The path through the notebooks:

1. **IIIF**: a shared standard that lets any viewer show images from any institution, at any size and detail.
2. **Europeana**: an aggregator with millions of descriptions from thousands of institutions, and an API. We treat it as a *dataset*, to find out what is inside.
3. **Semantic search**: a model (CLIP) that lets you search images by describing them in words.
4. **Regions and details**: what changes when you search with a part of an image instead of the whole.
5. **Your own contribution**: a short presentation, built from your own results.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 2 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

## How the notebooks are organised

A coloured banner starts every section. In this notebook the instructor first shows and explains; each section ends with a short task for you.

<div style="border-left:8px solid #BC8F8F;background:#ffffff;padding:12px 16px;border-radius:4px;margin:6px 0"><div style="font:600 12px sans-serif;color:#BC8F8F;letter-spacing:.08em">IIIF</div><div style="font:600 20px sans-serif;color:#1f2430">A section</div><div style="font:14px sans-serif;margin-top:4px;color:#1f2430">The instructor shows and explains. You follow along. At the end of the section: a task with cells to edit and run, and the solutions.</div></div><div style="border-left:8px solid #FF7F50;background:#ffffff;padding:12px 16px;border-radius:4px;margin:6px 0"><div style="font:600 12px sans-serif;color:#FF7F50;letter-spacing:.08em">EUROPEANA</div><div style="font:600 20px sans-serif;color:#1f2430">The next section</div><div style="font:14px sans-serif;margin-top:4px;color:#1f2430">The same pattern. Slides marked BROWSER happen in the browser, not in the notebook.</div></div><div style="border-left:8px solid #C0C0C0;background:#ffffff;padding:12px 16px;border-radius:4px;margin:6px 0"><div style="font:600 12px sans-serif;color:#C0C0C0;letter-spacing:.08em">APPENDIX</div><div style="font:600 20px sans-serif;color:#1f2430">Background</div><div style="font:14px sans-serif;margin-top:4px;color:#1f2430">Details, tables and references for reading afterwards. Not part of the session.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 3 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

## Working in a notebook

- A notebook is a sequence of **cells**. Text cells explain; code cells run Python.
- Click a cell and press **Shift+Enter** to run it. The result appears below it.
- Run cells **in order, from the top**. Later cells use what earlier ones defined.
- Select the kernel **DH Workshop** (top right). If a cell fails with a `NameError`, run the cells above it first.
- You cannot break anything. If a notebook gets into a muddle, use **Kernel → Restart Kernel**, then run from the top.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#708090"><span>WELCOME · 4 / 4</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #708090;margin:6px 0 0 0">

## Check your setup

Run the cell below. Each line should start with ✓. If something does not, tell the instructor now.

In [ ]:
import importlib, platform, sys
from pathlib import Path

def check(label, ok, detail=""):
    print(("✓" if ok else "✗"), label, detail)

check("Python", sys.version_info >= (3, 10), platform.python_version())
for pkg in ["requests", "PIL", "ipywidgets", "numpy", "torch", "clip"]:
    try:
        importlib.import_module(pkg)
        check(f"package {pkg}", True)
    except Exception as e:
        check(f"package {pkg}", False, f"({type(e).__name__})")

import requests
for name, url in [("IIIF server (iiif.io)", "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json"),
                  ("Europeana API", "https://api.europeana.eu/record/v2/search.json?wskey=api2demo&query=*&rows=0")]:
    try:
        check(name, requests.get(url, timeout=15).ok)
    except Exception as e:
        check(name, False, f"({type(e).__name__})")

root = Path(".").resolve().parent
check("dataset folder (semantic search)", (root / "data" / "DSI-Workshop").exists(), str(root / "data" / "DSI-Workshop"))

<div style="border-left:14px solid #BC8F8F;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#BC8F8F;letter-spacing:.08em">IIIF</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Images that work across institutions</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">Start in the browser, then look behind the scenes. The section ends with a short task.</div></div>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 1 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

# IIIF in practice

IIIF (the International Image Interoperability Framework, pronounced "triple-eye-eff") is a set of shared specifications for describing and serving images over the web.

**In this section**

1. A terapixel image, and what your browser requests
2. What IIIF is and where you find it
3. A Book of Hours, and how a manifest is nested
4. An annotated manuscript
5. How it works behind the scenes
6. Hands-on: change an image by editing its URL

In [ ]:
# Setup: run this cell first.
from iiif_helpers import *

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 2 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Start: a terapixel image, and what your browser requests

The [Terapixel panorama](https://terapixelpanorama.ch/) of the Battle of Murten is far too large to download at once. Yet it opens in a second and you can zoom to the brush strokes. How? Look at what the browser asks for.

1. Open the panorama: <https://terapixelpanorama.ch/>
2. Open the **developer tools**: press **F12** (or **Ctrl+Shift+I**; on a Mac **Cmd+Option+I**).
3. Click the **Network** tab. Clear the list (the 🚫 button).
4. Filter by **Img** (in Firefox: **Images**).
5. **Pan and zoom** in the panorama and watch the list fill up.
6. Click one request. The **Headers** tab shows its URL, the **Preview** tab the picture that came back.

**What to look for**

- Many small requests, each only a few KB, instead of one huge file.
- Zooming does not make a request larger: the viewer asks for *different* pieces.
- The URL says what it wants. A request from this panorama's own page looks like this (your numbers will differ):

```
https://murtenpanorama.epfl.ch/iiif/part123_decurve_align_crop_20240620_75_512.tif/1006421,318059,35839,47080/full/0/default.jpg
 \______________________ image service ______________________________________/ \_______ region _______/ size rot quality
```

The region is `x,y,width,height` in pixels of the full image. That is the whole trick, and it is what IIIF standardises.

![Detail of the Murten panorama by Louis Braun](../misc/assets/img/murten_panorama_detail.jpg)

<small>Detail of the panorama painting of the Battle of Murten by Louis Braun, shown at Expo.02. Photo: Riccardo Speziari, [Wikimedia Commons](https://de.wikipedia.org/wiki/Datei:Expo_02_-_Ausschnitt_Panoramagem%C3%A4lde_Schlacht_bei_Murten_-_Louis_Braun.jpg), CC BY-SA 3.0.</small>

*Starting from the Network tab is inspired by the IIIF introduction video tutorials of the Zentralbibliothek Zürich, made in the [ZB-Lab](https://www.zb.uzh.ch/en/ueber-uns/zb-lab).*

In [ ]:
url_panel("Terapixel panorama, Battle of Murten", [
    ("Terapixel panorama", "https://terapixelpanorama.ch/", "Open it, press F12, Network tab, filter by Img, then pan and zoom."),
    ("ZB-Lab (video tutorials on IIIF)", "https://www.zb.uzh.ch/en/ueber-uns/zb-lab", "The introduction that inspired this start."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 3 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## What is IIIF?

**International Image Interoperability Framework**

- A shared, open **standard** for delivering image-based resources on the web (and audio, video and 3D), so that they work across institutions.
- Also an open, global **community** of libraries, archives, museums, universities, developers and researchers who develop the specifications, build them into software and publish their collections.
- In short: images and their descriptions are offered through open interfaces (**APIs**), and any compatible tool can use them.

*Structure of this introduction after: Loosli, U. (2025), "IIIF – Der Standard für digitales Bildmaterial", AGIK-Workshop, 2 September 2025, Universitätsbibliothek Bern. Diagram and examples are our own.*

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 4 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Why does it matter?

![IIIF connects institutions, through open APIs, to any viewer or tool](../misc/assets/img/iiif_why.svg)

- The images **stay at the institution**. Nothing is copied or re-hosted.
- **One viewer** can show objects from many places side by side.
- Anyone can compare, annotate and reuse objects, because the interfaces are the same everywhere.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 5 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Where do I find IIIF?

<img src="../misc/assets/img/iiif_logo.png" alt="IIIF logo" align="right" width="130">

On many publication platforms. Some show this **IIIF logo**, or a link called **IIIF manifest**, on the object page. Others hide it in a menu, or do not show it at all (appendix A explains how to find the manifest anyway).

- [e-rara](https://www.e-rara.ch/) and [e-codices](https://www.e-codices.ch/): Swiss rare prints and manuscripts
- [e-manuscripta](https://www.e-manuscripta.ch/): manuscripts and archival material
- [Gallica](https://gallica.bnf.fr/): Bibliothèque nationale de France
- [Europeana](https://www.europeana.eu/): the European aggregator (next notebook)
- [Biblissima](https://portail.biblissima.fr/): a portal for medieval and early modern sources

<small>IIIF logo: International Image Interoperability Framework, public domain (text logo), via [Wikimedia Commons](https://commons.wikimedia.org/wiki/File:International_Image_Interoperability_Framework_logo.png).</small>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 6 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## A Book of Hours with many owners

[Fragmentarium](https://fragmentarium.ms/) record F-mp8l is a *virtual reconstruction* of a Book of Hours (France, early 15th century) that was taken apart in the 1940s. Its leaves are held by many institutions: Yale, Kent, Cleveland Public Library and more.

One manifest brings them back together: **61 canvases**, each with an image that stays on its owner's server, and **16 ranges** that rebuild the structure of the book (the Hours of the Virgin, the Hours of the Holy Spirit, ...).

Open the **raw JSON** of the manifest in a browser tab (below) and unfold it: `sequences` → `canvases` → one canvas → `images` → `resource` → `service`. The next slide is the same tree, as a diagram.

In [ ]:
BOH_MANIFEST = "https://fragmentarium.ms/metadata/iiif/F-mp8l/manifest.json"

url_panel("A Book of Hours, reassembled", [
    ("Fragmentarium record (reconstruction)", "https://fragmentarium.ms/overview/F-mp8l"),
    ("Single leaf, State Library of New York", "https://fragmentarium.ms/view/page/F-mp8l/7653/61324"),
    ("IIIF manifest = the raw JSON", BOH_MANIFEST, "Open it in a browser tab (Firefox shows JSON as an unfoldable tree). Or paste it into Mirador: Add resource."),
])

In [ ]:
# e-manuscripta: the same idea on a Swiss platform. Look for the IIIF icon on an object page: it links the manifest.
E_MANUSCRIPTA = "https://www.e-manuscripta.ch/"
E_MANUSCRIPTA_MANIFEST = None        # <-- the manifest URL of the object you show (the raw JSON), e.g. copied from its IIIF icon

url_panel("e-manuscripta", [("e-manuscripta", E_MANUSCRIPTA, "Manuscripts and archival material, Switzerland."),
                            *([("IIIF manifest = the raw JSON", E_MANUSCRIPTA_MANIFEST, "Open it in a browser tab and compare its nesting with the Book of Hours.")]
                              if E_MANUSCRIPTA_MANIFEST else [])])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 7 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

### How a manifest is nested

![IIIF structure: collection, manifest, ranges, canvases, images](../misc/assets/img/iiif_structure.svg)

- <b style="color:rosybrown">Manifest</b>: one object, one URL. In the Book of Hours: the whole reconstruction.
- <b style="color:crimson">Canvas</b>: a virtual page with width and height. List order is page order. Here: one leaf.
- <b style="color:blueviolet">Image</b>: attached to a canvas by a `painting` annotation. It can be hosted anywhere: every leaf on its owner's server.
- **Range**: a section (a table of contents) that groups canvases *without* reordering them.

In [ ]:
# The same tree as in the browser, printed: keys, list lengths, the first element of each list.
boh = get_json(BOH_MANIFEST)
show_outline(boh)

In [ ]:
if E_MANUSCRIPTA_MANIFEST:
    show_outline(get_json(E_MANUSCRIPTA_MANIFEST))

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 8 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## An annotated manuscript

A page of a digitised book can carry more than its image. On e-rara and e-manuscripta, a page has a **text layer**: a transcription, stored separately from the image and pointing at it. That is the idea of an **annotation**: a note that is about a page (or a part of it) without changing it.

This manuscript is one example: <https://www.e-rara.ch/bes_1/content/pagetext/32643449>

**The story**

- **The woman:** [Juliane von Krüdener](https://de.wikipedia.org/wiki/Juliane_von_Kr%C3%BCdener) (1764–1824), born in Riga into the Baltic German nobility, married at 17 to a Russian diplomat. After a religious conversion in 1804 she became a Pietist preacher and *prophetess*, travelled through south-west Germany, Alsace and Switzerland, and gained influence even over Tsar Alexander I. A noblewoman preaching in public: this shocked her contemporaries.
- **Her fate:** she was expelled from one state after another (Württemberg, Baden, Basel) and ended up in Russia, where she died in 1824 in Crimea. Some thought the treatment too harsh.
- **The man:** Friedrich von Dillenburg defends her in this manuscript. *(His role and why he writes: to be added.)*
- **Why it is here:** the defence is *text about a person*, preserved with the page images. Reading it takes a human. What would a program do with it?


In [ ]:
url_panel("Text layers on e-rara and e-manuscripta", [
    ("The annotated manuscript (page text, e-rara)", "https://www.e-rara.ch/bes_1/content/pagetext/32643449", "The transcription of the page, separate from the image."),
    ("Another page text: a handwritten note by Emmi Ball-Hennings (e-manuscripta, Swiss National Library)", "https://www.e-manuscripta.ch/snl/content/pagetext/4317519", "Open it to see the transcription as plain text."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 9 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How it works behind the scenes

Two specifications do the work, and each has its own kind of URL.

| API | Question it answers | What you get |
|---|---|---|
| **Presentation API** | What is this object, and how is it organised? | A JSON *manifest* |
| **Image API** | Give me this image, in this form. | Pixels, selected by a URL |

Below: the manifest of a small book (five pages, BnF / Gallica, from the IIIF Cookbook), and the URL of one image from it. Open both in a browser tab. The manifest contains the base URL of the image server; the image URL is built from it.


In [ ]:
BOOK_URL = "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json"
book = get_json(BOOK_URL)
PAGE = book["items"][2]                   # the title page
_, SERVICE = canvas_image(PAGE)
info = get_json(f"{SERVICE}/info.json")   # what the server can do
W, H = info["width"], info["height"]

url_panel("One object, one image", [
    ("Presentation API: the manifest (JSON)", BOOK_URL, "Open it in a browser tab: title, canvases in order, and for each the address of its image."),
    ("Image API: the title page as an image", iiif_url(SERVICE, size="600,"), "The counterpart: pixels, delivered by a different server."),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 10 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## One image, any view

An image service answers URLs of a fixed form:

```
{service}/{region}/{size}/{rotation}/{quality}.{format}
```

![Anatomy of an IIIF image URL](../misc/assets/img/iiif_image_url.svg)

In [ ]:
show_views(SERVICE, [
    ("full", "full", "260,", "0", "default"),
    ("a region", f"{W*28//100},{H*45//100},{W*33//100},{H*23//100}", "260,", "0", "default"),
    ("rotated 90°", "full", ",260", "90", "default"),
    ("grey", "full", "260,", "0", "gray"),
])

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 11 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Your turn: change an image by editing its URL

*Optional, in your own browser, at your own pace.*

Every view of an image is just a URL. Open the image below in a browser tab, then change one segment of the address in the address bar and press Enter.

![Anatomy of an IIIF image URL](../misc/assets/img/iiif_image_url.svg)

- <b style="color:rosybrown">region</b>: `full`, `square`, `x,y,w,h` (pixels), `pct:x,y,w,h` (percent)
- <b style="color:coral">size</b>: `max`, `400,` (width), `,300` (height), `!400,300` (fit inside), `pct:50`
- <b style="color:crimson">rotation</b>: `0`, `90`, `180`, `270`, `!0` (mirrored)
- <b style="color:blueviolet">quality</b> and <b style="color:blueviolet">format</b>: `default`, `gray`, `bitonal`; `.jpg`, `.png`


In [ ]:
url_panel("The demo image from 'One image, any view'", [
    ("Image URL", iiif_url(SERVICE, size="600,"), "Open it in a browser tab, then edit the segments in the address bar."),
])

### Tasks

1. **Rotate** the image by 90°, then 180°.
2. **Mirror** it.
3. Make it **grey**, then **bitonal** (black and white only).
4. **Crop** to the circular vignette: region `900,2100,1050,1050`. Then try the percentage form: `pct:25,40,35,25`.
5. Request a **small thumbnail**, 150 pixels wide.
6. **Combine**: a grey, rotated, cropped detail.


### Stretch

- Add `/info.json` to the service URL (open it in the browser). Read `width` and `height`, then request the **bottom-right quarter** of the image.
- Open the manifest of the Book of Hours in Mirador and find the image service URL of one leaf in the manifest. Does it work with the same URL pattern?

<details><summary><b>Solutions</b> (click to open)</summary>

1. rotation `90`, then `180`
2. rotation `!0`
3. quality `gray`, then `bitonal`
4. region `900,2100,1050,1050`, or `pct:25,40,35,25`
5. size `150,`
6. for example region `900,2100,1050,1050`, size `400,`, rotation `90`, quality `gray`
- Bottom-right quarter: region `pct:50,50,50,50`
</details>

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 12 / 13</span>&nbsp;&nbsp;</div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## How do I use it?

1. Find the **manifest link** (or the IIIF logo) on the object page.
2. **Copy** the link.
3. Open a IIIF viewer, for example [Mirador](https://projectmirador.org/demo/) or [Universal Viewer](https://universalviewer.io/).
4. **Paste** the link. The object appears in the viewer.

In Mirador you can place several objects side by side to compare them, and a workspace can be exported and imported again. Many IIIF tools work with images without downloading and uploading them: for example [Tropy](https://tropy.org/) for archival material, or eScriptorium and Transkribus for text recognition. A long list: <https://iiif.io/get-started/tools/>.

<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#BC8F8F"><span>IIIF · 13 / 13</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #BC8F8F;margin:6px 0 0 0">

## Mirador: paste in a manifest

[Mirador](https://projectmirador.org/) is given only the manifest URL. It works out page order, thumbnails, deep zoom and the table of contents by itself.

**Your turn** *(about 5 minutes)*

1. Open one of the sites from *Where do I find IIIF?* (not Europeana: finding its manifest is harder, see the appendix).
2. Pick an object and find its **manifest URL**: look for the IIIF logo or a link called *IIIF* or *Manifest*, often in a share, download or export menu.
3. Paste it into `MY_MANIFEST` below and run the cell. Or paste it into the [Mirador demo](https://projectmirador.org/demo/) with **Add resource**.


In [ ]:
MY_MANIFEST = BOOK_URL        # <-- replace with the manifest URL you found, in quotes
show_mirador(MY_MANIFEST)

<div style="border-left:14px solid #C0C0C0;background:#ffffff;padding:70px 40px;border-radius:4px"><div style="font:600 20px sans-serif;color:#C0C0C0;letter-spacing:.08em">APPENDIX</div><div style="font:700 48px sans-serif;margin-top:10px;color:#1f2430">Background</div><div style="font:22px sans-serif;margin-top:16px;color:#1f2430">Details for reading afterwards. Not part of the presentation.</div></div>

## IIIF · A. Finding a manifest

Not every page shows the IIIF logo, and where it does, it may sit in a menu. Four ways to get to the manifest, from the easiest:

| | Strategy | Example |
|---|---|---|
| 1 | **Build it from the page address or ID**, where the platform follows a pattern | Europeana: item `.../item/90402/RP_T_1904_190` gives `iiif.europeana.eu/presentation/90402/RP_T_1904_190/manifest` (dataset *and* record). e-codices: `.../metadata/iiif/{shelfmark}/manifest.json`. Fragmentarium: `.../metadata/iiif/{id}/manifest.json` |
| 2 | **Look on the object page** for the IIIF logo or a link called *IIIF* or *Manifest*, often in a share, download or export menu | e-rara, e-manuscripta, Gallica |
| 3 | **Ask the browser**: developer tools, **Network** tab, reload the page, filter by `manifest` (the same tool as for the panorama). Or view the page source and search for `manifest` | any platform |
| 4 | **Look it up** in the community guide [Finding IIIF Resources](https://guides.iiif.io/finding_resources/), or on a portal such as [Biblissima](https://portail.biblissima.fr/) | any platform |

`find_manifest(url)` does 1, and a version of 3 (it searches the page text for addresses with `manifest`). It requests every candidate and only returns one that really is a manifest. Where it finds nothing, use 2 and 3 by hand.

In [ ]:
examples = [
    "https://www.europeana.eu/en/item/1387/https___id_kulturpool_at_013cf817_e57a_4289_96b3_2a068c7fe756_cho",    # 1: Europeana
    "https://www.europeana.eu/en/item/90402/RP_T_1904_190",                                                      # 1: Europeana, a Rijksmuseum drawing
    "https://www.e-codices.unifr.ch/en/csg/0001/",                                                                 # 1: e-codices
    "https://fragmentarium.ms/overview/F-mp8l",                                                                    # 1: Fragmentarium
    "https://iiif.io/api/cookbook/recipe/0009-book-1/",                                                            # 3: the page text mentions it
]
for page in examples:
    print(page)
    find_manifest(page)
    print()

YOUR_PAGE = ""        # <-- the address of an object page you found, from any platform
if YOUR_PAGE:
    find_manifest(YOUR_PAGE)

## IIIF · B. Two APIs, two kinds of URL

![From record ID to image request: the manifest URL, the lookup inside the manifest, the image URLs](../misc/assets/img/iiif_url_translation.svg)

| | Presentation API | Image API |
|---|---|---|
| **Identifies** | one **object** (a book, a painting, ...) | one **image** |
| **Its URL** | the **manifest** | the **base URL** of an image service |
| **Gives you** | JSON: title, rights, canvases in order | `{base}/info.json` (size, capabilities), or pixels: `{base}/{region}/{size}/{rotation}/{quality}.{format}` |

**How they connect.** The manifest holds the base URL of each image: canvas, painting annotation, body, `service`, `id`. A manifest URL can often be *built* from a page address. The image base URL is always *read from the manifest*: the image server may belong to another institution, with its own identifiers.

**Europeana, worked example.** The item page `https://www.europeana.eu/en/item/90402/RP_T_1904_190` has two identifiers: the dataset (`90402`) and the record (`RP_T_1904_190`). Both go into the manifest URL:

`https://iiif.europeana.eu/presentation/90402/RP_T_1904_190/manifest` (the ending `.json` is optional)

Leave out the dataset number and the server answers with an error. And a manifest does not promise an image service: this one names a plain image file, so there is no Image API behind it.

In [ ]:
# A complete chain on e-codices: manifest, canvas, image file, image service (on another server), info.json, a region request.
show_chain("https://www.e-codices.unifr.ch/metadata/iiif/csg-0001/manifest.json")

In [ ]:
# The Europeana record from above: the manifest works, the chain ends at a plain image file.
show_chain("https://iiif.europeana.eu/presentation/90402/RP_T_1904_190/manifest")

## IIIF · C. A manifest

A small book from the IIIF Cookbook: five pages digitised by the BnF (Gallica). The manifest is a few kilobytes of JSON; the images come from a separate image server.

In [ ]:
book = get_json(BOOK_URL)
print("label:", first(book["label"]))
print("pages:", len(book["items"]))

thumb_strip(book["items"])

## IIIF · D. The structure of a manifest in more detail

- <b style="color:darkgray">Collection</b> is a list of manifests (or of other collections).
- <b style="color:rosybrown">Manifest</b> describes one object, for example a book. It carries the title, rights statement, attribution and other metadata, and it is the unit you share: one URL stands for the whole object.
- <b style="color:crimson">Canvas</b> is a virtual page: an empty rectangle with a width, a height and a label such as "fol. 1r". Canvases are listed in reading order, and that list order *is* the page order.
- <b style="color:blueviolet">Image</b> is not part of the canvas itself. It is attached to the canvas by an annotation with the motivation <code style="color:blueviolet; border:1.5pt solid blueviolet; padding:1px 2px; border-radius:5px;background:#FFFFFF">painting</code>, which says "place this image onto this canvas". The image can be hosted anywhere, so neighbouring pages of one book may come from different servers, as in the diagram (Yale, Smith College, BnF).
- <b style="color:coral">Range</b> is an optional grouping of canvases, such as a chapter or a section. Ranges give a table of contents without changing the canvas order.

The separation between canvas and image is what lets a manifest organise pages without owning any pixels.

## IIIF · E. Following the chain from canvas to image

`items` holds the canvases in reading order. For every canvas we can follow *canvas → annotation page → painting annotation → image body → image service*:

In [ ]:
rows = ""
for i, cv in enumerate(book["items"], 1):
    body, svc = canvas_image(cv)
    rows += (f"<tr><td>{i}</td><td>{html.escape(first(cv['label']))}</td>"
             f"<td>{cv['width']} × {cv['height']}</td><td><code>…{svc[-24:]}</code></td></tr>")
display(HTML(
    "<table><tr><th>#</th><th>canvas label</th><th>canvas size</th><th>image service (end of URL)</th></tr>"
    + rows + "</table>"))

cv = book["items"][2]
body, svc = canvas_image(cv)
trimmed = {
    "id": cv["id"], "type": "Canvas", "label": cv["label"],
    "width": cv["width"], "height": cv["height"],
    "items": [{"type": "AnnotationPage", "items": [{
        "type": "Annotation", "motivation": "painting",
        "body": {"type": "Image", "service": [{"id": svc, "type": "ImageService3"}]},
        "target": cv["id"],
    }]}],
}
print(json.dumps(trimmed, indent=2))

- `width` and `height` on the canvas define an abstract coordinate space, the reference for everything placed on the canvas, including annotations.
- The painting annotation's `target` is the canvas, and its `body` is the image. That is the whole link between the two.
- The only thing the manifest knows about the pixels is the address of an image service.

## IIIF · F. The Image API in more detail

`info.json` sits next to every image service and states what the server can do. `width` and `height` give the size of the original in pixels, and `tiles` describes how a viewer may cut it into pieces.

In [ ]:
print(json.dumps(info, indent=2))

### Same image, different sizes

Four requests to the same service, changing only the `size` segment. The smallest version is several hundred times smaller than the full-size file, and it was produced on request: the institution only needs to keep one high-resolution master.

In [ ]:
sizes = [("100 px wide", "100,"), ("400 px wide", "400,"), ("1000 px wide", "1000,"), ("full size", "max")]
results = []
for label, size in sizes:
    url = f"{SERVICE}/full/{size}/0/default.jpg"
    im, n = pil(url)
    results.append((label, url, im, n))

display(HTML("<table><tr><th>request</th><th>pixels returned</th><th>transferred</th></tr>" + "".join(
    f"<tr><td><code>full/{u.split('/full/')[1].split('/')[0]}/0/default.jpg</code></td>"
    f"<td>{im.width} × {im.height}</td><td>{kb(n)}</td></tr>" for _, u, im, n in results) + "</table>"))

### Tiles and scale factors

`info.json` advertises a tile size (`512`) and a list of `scaleFactors` (`1, 2, 4, 8`), which describe zoom levels:

| scale factor | meaning | tile covers (in original pixels) |
|---|---|---|
| 8 | image shown at 1/8 size, most zoomed out | 4096 × 4096 |
| 4 | 1/4 size | 2048 × 2048 |
| 2 | 1/2 size | 1024 × 1024 |
| 1 | full resolution, most zoomed in | 512 × 512 |

At every level a tile is a 512-pixel-wide picture, so the cost of filling the screen stays roughly constant however large the original is. The function `render_viewport` in `iiif_helpers.py` does what a viewer does: it works out which tiles overlap a 512 × 512 pixel window, requests only those, and joins them together.

- Servers advertise a **compliance level** in `info.json` (`profile`). Level 0 only offers pre-cut tiles, while levels 1 and 2 accept arbitrary regions and sizes.
- IIIF image services exist in versions 2 and 3, which differ in small ways (for example `max` replaces `full` as the size keyword). Requests such as `,400` work on both.

## IIIF · G. Deep zoom with tiles

A viewer never loads the whole image. It asks for **tiles**: ordinary region requests at a fixed scale. The same 512 × 512 pixel window, at four zoom levels, built from a few tiles each:

In [ ]:
deep_zoom_demo(SERVICE, info, focus=(1425, 2640))

## IIIF · H. Notes on a region of a page

A canvas has a coordinate space, so a note can point at a *part* of it. The target is the canvas URL plus `#xywh=x,y,w,h`. The note is separate JSON; the image is not changed.

In [ ]:
REGIONS = [
    ((900, 2100, 1050, 1050), "Circular vignette showing a figure in a tree."),
    ((230, 1520, 510, 500), "Red library stamp."),
]
annotation_page = {
    "@context": "http://iiif.io/api/presentation/3/context.json",
    "id": "https://example.org/annotations/title-page",
    "type": "AnnotationPage",
    "items": [make_annotation(PAGE["id"], r, t, i + 1) for i, (r, t) in enumerate(REGIONS)],
}
print(json.dumps(annotation_page["items"][0], indent=2))
overlay(PAGE, SERVICE, annotation_page)

## IIIF · I. Annotations: coordinates and canvas size

Reading an annotation back is the reverse operation: take the region from the target and use it in an Image API request to retrieve exactly the annotated pixels. In this manifest the canvas and the image have the same dimensions. In general they can differ, so the conversion multiplies by the ratio of image size to canvas size.

In [ ]:
body, _ = canvas_image(PAGE)
show_annotated_regions(annotation_page, PAGE, body, SERVICE)

## IIIF · J. IIIF in Switzerland

IIIF is widely used in Swiss cultural heritage institutions. The overview below follows the report by Julien A. Raemy, *IIIF in Switzerland: A Landscape of Compatible Platforms and Engaged Institutions* (see the source at the end of this section). It surveys platforms that meet explicit criteria: valid IIIF resources, operation by a public service or research institution, and a Swiss operator or Swiss holdings. Figures are as of October 2026.

- **23 platforms** are included: 17 publish Presentation API resources (manifests), and 6 qualify through their Image API services alone. Together they publish well over 600,000 manifests, counting only the platforms for which a figure could be estimated.
- **Libraries started it.** In 2019 IIIF in Switzerland was largely a library matter. The long-standing portals e-codices, e-rara, e-manuscripta, e-newspaperarchives.ch and Fragmentarium still use version 2.x of the APIs. e-codices was the first platform in the world to implement both the Image API 2.0 and the Presentation API 2.0 (2014).
- **Archives and museums have joined.** The Swiss Federal Archives publish Presentation API 3.0 and support the Content Search API. Museum platforms such as SKKG (Winterthur) publish more than 60,000 manifests in version 3.0.
- **Version 3.0 is still the minority.** Seven platforms use Presentation API 3.0, while ten, among them the long-standing library portals, remain on 2.x.
- **Shared infrastructure.** Research projects can publish IIIF resources without running servers, for example through Hedera (University of Geneva), which hosts the more than 100,000 manifests of the Visual Contagions project.
- **Swiss members of the IIIF Consortium:** DaSCH (the national data service for the humanities, which develops the open source image server SIPI) and the University of Bern.
- **Discoverability lags behind adoption.** Only 6 of the 23 platforms are listed in the community guide *Finding IIIF Resources*, and few advertise a top-level collection.

| Platform | Operator | Content | Presentation API | Manifests |
|---|---|---|---|---|
| e-codices | University of Fribourg | Medieval and early modern manuscripts | 2.x | > 3,000 |
| e-manuscripta | Zentralbibliothek Zürich and partners | Manuscripts and archival material | 2.x | ≈ 200,000 |
| e-rara | ETH Library and partners | Printed books | 2.x | ≈ 150,000 |
| Fragmentarium | Fragmentarium, Fribourg | Medieval manuscript fragments | 2.x | > 8,000 |
| Minutes of the Federal Council | Swiss Federal Archives | Minutes, 1848 to 1975 | 3.0 | > 10,000 |
| SKKG, Digital Collection | Stiftung für Kunst, Kultur und Geschichte | Art, culture and history | 3.0 | > 60,000 |
| Visual Contagions (via Hedera) | University of Geneva | Circulation of images in printed media | 3.0 | > 100,000 |

Two platforms in this table appear in this notebook: Fragmentarium (the Book of Hours) and e-manuscripta (the annotated manuscript). Manifest counts are rounded by the author (">" is a lower bound, "≈" an approximation).

**Source:** Raemy, J. A. (2026). *IIIF in Switzerland: A Landscape of Compatible Platforms and Engaged Institutions* (Version 1.0) [Report]. University of Bern. <https://doi.org/10.5281/zenodo.23242984>. Licence: CC BY 4.0. The text above is summarised from the report; the table is a selection from its Table 1.

## IIIF · K. Summary

| What the standard provides | How |
|---|---|
| One address per object | The manifest URL identifies a whole book, with its metadata and rights |
| Ordered pages, independent of file names | A list of canvases; each canvas is a coordinate space |
| Images hosted anywhere | A painting annotation connects an image service to a canvas |
| Sections and tables of contents | Ranges group canvases without reordering them |
| Any size or region of an image on request | The Image API URL: `{service}/{region}/{size}/{rotation}/{quality}.{format}` |
| Deep zoom with small transfers | Tiles are ordinary region requests at fixed scale factors |
| Notes on part of a page | Annotations target `canvas#xywh=…` and leave the image untouched |
| Assembling one object from many sources | A manifest can list canvases whose images come from different services |

### Limits to keep in mind

- Not every institution offers an image service. Some only publish static image files, in which case region and size requests are not available. Check the `service` entry before relying on them.
- Servers differ in compliance level, in API version (2 or 3) and in the formats they support. Always read `info.json`.
- A manifest is a description made by someone. Its page order, labels and groupings reflect that person's decisions, which is also true of reconstructions such as the Book of Hours.

### References

- IIIF overview and specifications: <https://iiif.io/>
- Presentation API 3.0: <https://iiif.io/api/presentation/3.0/>
- Image API 3.0: <https://iiif.io/api/image/3.0/>
- IIIF Cookbook (worked manifest examples, including annotations): <https://iiif.io/api/cookbook/>
- Mirador: <https://projectmirador.org/>
- Fragmentarium: <https://fragmentarium.ms/>
- Loosli, U. (2025). *IIIF – Der Standard für digitales Bildmaterial* [Workshop slides, AGIK-Workshop, 2 September 2025]. Universitätsbibliothek Bern. Source of the structure of the introduction.


<div style="margin:1vh 0 0 0;font:600 12px sans-serif;letter-spacing:.1em;color:#C0C0C0"><span>APPENDIX · 1 / 1</span>&nbsp;&nbsp;<span style="border:2.2px solid #8A2BE2;border-radius:8px;background:#fff;color:#8A2BE2;padding:2px 10px;font:700 12px sans-serif;letter-spacing:.1em">BROWSER</span></div><hr style="border:0;border-top:1px solid #C0C0C0;margin:6px 0 0 0">

## IIIF · L. Extra demo: two sources in one viewer

Open the [Mirador demo](https://projectmirador.org/demo/), choose **Add resource** (the `+` button), paste a manifest URL and press **Add**. Do it twice.

- Two institutions, two servers, one viewer.
- Mirador gets nothing but a URL. Page order, thumbnails and zoom come from the manifest.

In [ ]:
# Replace these two with the panorama manifests (earlier state, more recent state).
EARLIER = "https://api.artic.edu/api/v1/artworks/143948/manifest.json"   # placeholder: Grindelwald glacier, 19th c.
RECENT  = "https://iiif.io/api/cookbook/recipe/0009-book-1/manifest.json" # placeholder: any second working manifest

# Third, most recent state: satellite view in Google Maps (Rhone glacier). Adjust the coordinates and zoom as needed.
MAPS = "https://www.google.com/maps/@46.6104,8.4025,3000m/data=!3m1!1e3"

url_panel("Demo A: URLs", [
    ("Mirador demo page", "https://projectmirador.org/demo/", "Open this first, then use 'Add resource' for each manifest below."),
    ("Manifest 1: earlier", EARLIER, "Paste into Add resource."),
    ("Manifest 2: more recent", RECENT, "Paste into Add resource, then open it in a second window."),
    ("Google Maps: today", MAPS, "Satellite view; the most recent state of the glacier."),
])

## Next

The next notebook, **02_europeana**, treats Europeana as a dataset: who contributes, what is inside, and how to ask its API. Records there lead back to IIIF manifests, like the ones in this notebook.